# Project 9: Image Classification (Basic Objects) — Fruit Classification
### Using Convolutional Neural Network (CNN)
**Dataset:** [Fruit Classification 10 Class](https://www.kaggle.com/datasets/karimabdulnabi/fruit-classification10-class)

## Step 1: Problem Statement
This project aims to classify images of fruits into 10 distinct categories using a Convolutional Neural Network (CNN). Automated fruit classification has practical applications in agriculture, food processing, and retail industry. Given an image, the model must predict which fruit class it belongs to.

## Step 2: Dataset Download & Setup

In [ ]:
# ── Install & import dependencies ──────────────────────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['kaggle', 'tensorflow', 'scikit-learn', 'matplotlib', 'seaborn', 'numpy', 'pillow']:
    install(pkg)

print('All packages installed successfully.')

In [ ]:
import os, zipfile, shutil, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from collections import Counter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score
)

warnings.filterwarnings('ignore')
tf.get_logger().setLevel('ERROR')

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# ── Download dataset from Kaggle ───────────────────────────────────────────────
# IMPORTANT: Place your kaggle.json in ~/.kaggle/ before running this cell.
# Get it from: https://www.kaggle.com/settings  →  API  →  Create New Token

KAGGLE_JSON_PATH = os.path.expanduser('~/.kaggle/kaggle.json')

if not os.path.exists(KAGGLE_JSON_PATH):
    # ── Fallback: create kaggle.json from environment variables ──────────────
    KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
    KAGGLE_KEY      = os.environ.get('KAGGLE_KEY', '')
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
        import json
        with open(KAGGLE_JSON_PATH, 'w') as f:
            json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
        os.chmod(KAGGLE_JSON_PATH, 0o600)
        print('kaggle.json created from environment variables.')
    else:
        print('WARNING: kaggle.json not found and environment variables not set.')
        print('Please place kaggle.json at ~/.kaggle/kaggle.json and re-run.')
else:
    print('kaggle.json found.')

os.makedirs('data', exist_ok=True)

ZIP_PATH = 'data/fruit-classification10-class.zip'

if not os.path.exists(ZIP_PATH):
    print('Downloading dataset ...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d',
         'karimabdulnabi/fruit-classification10-class',
         '-p', 'data', '--force'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
    else:
        print('Download complete.')
else:
    print('ZIP already exists, skipping download.')

In [ ]:
# ── Unzip dataset ──────────────────────────────────────────────────────────────
EXTRACT_DIR = 'data/fruits'

if not os.path.exists(EXTRACT_DIR):
    print('Extracting ZIP ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction complete.')
else:
    print('Already extracted.')

# ── Discover dataset structure ─────────────────────────────────────────────────
for root, dirs, files in os.walk(EXTRACT_DIR):
    depth = root.replace(EXTRACT_DIR, '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth >= 2:
        print(f'{indent}  [{len(files)} files]')
        dirs[:] = []  # stop deeper recursion

In [ ]:
# ── Locate train/test directories ─────────────────────────────────────────────
def find_split_dirs(base):
    """Search for train/test directories regardless of nesting."""
    train_dir = test_dir = None
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            name = d.lower()
            full = os.path.join(root, d)
            if name in ('train', 'training') and train_dir is None:
                train_dir = full
            elif name in ('test', 'testing', 'val', 'validation') and test_dir is None:
                test_dir = full
    return train_dir, test_dir

TRAIN_DIR, TEST_DIR = find_split_dirs(EXTRACT_DIR)
print(f'Train dir : {TRAIN_DIR}')
print(f'Test  dir : {TEST_DIR}')

# If no split found, use the root and split manually later
if TRAIN_DIR is None:
    # Flat structure — use first sub-directory that contains class folders
    candidates = [os.path.join(EXTRACT_DIR, d)
                  for d in os.listdir(EXTRACT_DIR)
                  if os.path.isdir(os.path.join(EXTRACT_DIR, d))]
    DATASET_ROOT = candidates[0] if candidates else EXTRACT_DIR
    print(f'No explicit split found. Using: {DATASET_ROOT}')
    TRAIN_DIR = TEST_DIR = DATASET_ROOT
else:
    DATASET_ROOT = os.path.dirname(TRAIN_DIR)

## Step 3: Data Preprocessing

In [ ]:
# ── Image parameters ────────────────────────────────────────────────────────────
IMG_SIZE    = (64, 64)   # resize all images to 64×64
BATCH_SIZE  = 32
SEED        = 42

# ── Normalise pixel values to [0,1] + augment training set ────────────────────
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.15,
    validation_split=0.2   # 80 % train, 20 % val when no explicit split
)

test_datagen = ImageDataGenerator(rescale=1./255)

# ── Load generators ────────────────────────────────────────────────────────────
if TRAIN_DIR != TEST_DIR:
    train_gen = train_datagen.flow_from_directory(
        TRAIN_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical', seed=SEED
    )
    test_gen = test_datagen.flow_from_directory(
        TEST_DIR, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical',
        shuffle=False, seed=SEED
    )
else:
    # Use validation_split to carve out 20 %
    train_gen = train_datagen.flow_from_directory(
        DATASET_ROOT, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical',
        subset='training', seed=SEED
    )
    test_gen = train_datagen.flow_from_directory(
        DATASET_ROOT, target_size=IMG_SIZE,
        batch_size=BATCH_SIZE, class_mode='categorical',
        subset='validation', shuffle=False, seed=SEED
    )

CLASS_NAMES  = list(train_gen.class_indices.keys())
NUM_CLASSES  = len(CLASS_NAMES)

print(f'Number of classes : {NUM_CLASSES}')
print(f'Class names       : {CLASS_NAMES}')
print(f'Training samples  : {train_gen.samples}')
print(f'Test/Val samples  : {test_gen.samples}')

## Step 4: Data Visualisation

In [ ]:
# ── Graph 1: Class distribution (Bar Chart) ───────────────────────────────────
class_counts = {}
for cls in CLASS_NAMES:
    folder = os.path.join(TRAIN_DIR if TRAIN_DIR != TEST_DIR else DATASET_ROOT, cls)
    if os.path.isdir(folder):
        class_counts[cls] = len([
            f for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ])
    else:
        class_counts[cls] = 0

palette = sns.color_palette('Set2', NUM_CLASSES)
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(class_counts.keys(), class_counts.values(),
              color=palette, edgecolor='black', linewidth=0.6)
ax.bar_label(bars, fmt='%d', padding=3, fontsize=9)
ax.set_title('Graph 1 — Class Distribution in Training Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Fruit Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_xticklabels(class_counts.keys(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('graph1_class_distribution.png', dpi=150)
plt.show()
print('Graph 1 saved.')

In [ ]:
# ── Graph 2: Sample images grid ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Graph 2 — Sample Images From Each Class', fontsize=14, fontweight='bold')

for idx, cls in enumerate(CLASS_NAMES[:10]):
    folder = os.path.join(TRAIN_DIR if TRAIN_DIR != TEST_DIR else DATASET_ROOT, cls)
    if os.path.isdir(folder):
        imgs = [f for f in os.listdir(folder)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if imgs:
            img_path = os.path.join(folder, imgs[0])
            img = mpimg.imread(img_path)
            r, c = divmod(idx, 5)
            axes[r, c].imshow(img)
            axes[r, c].set_title(cls, fontsize=10)
            axes[r, c].axis('off')

plt.tight_layout()
plt.savefig('graph2_sample_images.png', dpi=150)
plt.show()
print('Graph 2 saved.')

In [ ]:
# ── Graph 3: Pixel intensity histogram ────────────────────────────────────────
from tensorflow.keras.preprocessing.image import load_img, img_to_array

sample_pixels = []
for cls in CLASS_NAMES:
    folder = os.path.join(TRAIN_DIR if TRAIN_DIR != TEST_DIR else DATASET_ROOT, cls)
    if os.path.isdir(folder):
        files = [f for f in os.listdir(folder)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:10]
        for fname in files:
            arr = img_to_array(load_img(os.path.join(folder, fname),
                                        target_size=IMG_SIZE)) / 255.0
            sample_pixels.append(arr.flatten())

all_pixels = np.concatenate(sample_pixels)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(all_pixels, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Graph 3 — Pixel Intensity Distribution (Normalised)', fontsize=14, fontweight='bold')
ax.set_xlabel('Pixel Intensity (0–1)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
plt.tight_layout()
plt.savefig('graph3_pixel_histogram.png', dpi=150)
plt.show()
print('Graph 3 saved.')

In [ ]:
# ── Graph 4: Pie chart — train vs test split ──────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
sizes  = [train_gen.samples, test_gen.samples]
labels = ['Training Set', 'Test / Validation Set']
colors = ['#66b3ff', '#ff9999']
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=140, wedgeprops=dict(edgecolor='white', linewidth=1.5))
ax.set_title('Graph 4 — Dataset Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('graph4_dataset_split.png', dpi=150)
plt.show()
print('Graph 4 saved.')

## Step 5: Algorithm Implementation — CNN

In [ ]:
# ── Build CNN model ────────────────────────────────────────────────────────────
def build_cnn(input_shape, num_classes):
    model = keras.Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), activation='relu', padding='same',
                      input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 2
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.25),

        # Block 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2,2),
        layers.Dropout(0.4),

        # Classifier head
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ], name='FruitCNN')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model = build_cnn((*IMG_SIZE, 3), NUM_CLASSES)
model.summary()

## Steps 6 & 7: Model Training and Testing

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss', patience=8,
    restore_best_weights=True, verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=4, min_lr=1e-6, verbose=1
)

# ── Train ──────────────────────────────────────────────────────────────────────
EPOCHS = 30

history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print('\nTraining complete.')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='steelblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='tomato')
axes[0].set_title('Graph 5 — Model Accuracy Over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='tomato')
axes[1].set_title('Graph 6 — Model Loss Over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('graph5_6_training_curves.png', dpi=150)
plt.show()
print('Training curves saved.')

## Step 8: Performance Evaluation

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

# ── Accuracy ──────────────────────────────────────────────────────────────────
acc = accuracy_score(y_true, y_pred)
print(f'\n=== Performance Evaluation ===')
print(f'Test Accuracy : {acc*100:.2f}%')

# ── Precision & Recall ────────────────────────────────────────────────────────
prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
print(f'Precision     : {prec:.4f}')
print(f'Recall        : {rec:.4f}')

# ── Classification Report ─────────────────────────────────────────────────────
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, ax=ax)
ax.set_title('Graph 7 — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted Class', fontsize=12)
ax.set_ylabel('True Class', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('graph7_confusion_matrix.png', dpi=150)
plt.show()
print('Confusion matrix saved.')

In [ ]:
# ── Per-class precision & recall bar chart ────────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support

prec_per, rec_per, f1_per, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0
)

x = np.arange(NUM_CLASSES)
w = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w/2, prec_per, w, label='Precision', color='steelblue')
ax.bar(x + w/2, rec_per,  w, label='Recall',    color='tomato')
ax.set_title('Graph 8 — Per-Class Precision & Recall', fontsize=14, fontweight='bold')
ax.set_xlabel('Fruit Class', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('graph8_precision_recall.png', dpi=150)
plt.show()
print('Precision-Recall chart saved.')

## Step 9: Result Analysis

In [ ]:
# ── Print final summary ───────────────────────────────────────────────────────
best_class = CLASS_NAMES[np.argmax(prec_per)]
worst_class = CLASS_NAMES[np.argmin(prec_per)]

print('=' * 55)
print('           RESULT ANALYSIS SUMMARY')
print('=' * 55)
print(f'  Overall Test Accuracy    : {acc*100:.2f}%')
print(f'  Weighted Precision       : {prec:.4f}')
print(f'  Weighted Recall          : {rec:.4f}')
print(f'  Best Classified Class    : {best_class}')
print(f'  Hardest Classified Class : {worst_class}')
print('=' * 55)
print()
print('Analysis:')
print(f'  The CNN model achieved {acc*100:.1f}% accuracy on the test set.')
print(f'  {best_class} had the highest precision, indicating that images')
print(f'  of this fruit are highly distinctive and easy to separate.')
print(f'  {worst_class} was the most challenging class, likely due to')
print('  visual similarity with other classes or limited sample variation.')
print()
print('  The use of BatchNormalization, Dropout, and data augmentation')
print('  helped regularize the model and reduce overfitting.')

## Step 10: Conclusion

In [ ]:
print('=' * 55)
print('                   CONCLUSION')
print('=' * 55)
print()
print('This project successfully implemented a Convolutional')
print('Neural Network (CNN) to classify 10 types of fruits.')
print()
print('Key Findings:')
print(f'  • Test Accuracy  : {acc*100:.2f}%')
print(f'  • Precision      : {prec:.4f}')
print(f'  • Recall         : {rec:.4f}')
print()
print('The CNN leveraged spatial feature extraction through')
print('convolutional layers, and generalisation was improved')
print('via dropout, batch normalisation, and data augmentation.')
print()
print('Future improvements could include:')
print('  • Transfer learning (e.g., MobileNetV2, ResNet50)')
print('  • Higher-resolution inputs (128×128 or 224×224)')
print('  • Expanding the dataset with more augmentation')
print('=' * 55)

In [ ]:
# ── Save trained model ────────────────────────────────────────────────────────
model.save('fruit_classifier_cnn.h5')
print('Model saved to fruit_classifier_cnn.h5')